# Imports
This cell imports the required libraries for data manipulation, model training, cross-validation, and evaluation.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, GridSearchCV

# Load dataset
Read the dataset from a CSV file into a pandas DataFrame. The encoding option is used to avoid decoding errors on some files.

In [7]:
df = pd.read_csv('../mushroom/mushrooms.csv', encoding='unicode_escape')
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


# One-hot encoding for mushroom dataset
Encode all categorical feature columns (everything except the target `class`) using one-hot encoding so they can be used by GaussianNB.

In [8]:
# One-hot encode all feature columns (exclude the target 'class')
df_features = df.drop('class', axis=1)
df3 = pd.get_dummies(df_features, drop_first=False)
# df3 holds the encoded features; copy to df4 for compatibility with downstream cells
df4 = df3.copy()
df3.head()

,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_f,cap-surface_g,cap-surface_s,cap-surface_y,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,False,False,False,False,False,True,False,False,True,False,...,True,False,False,False,False,False,False,False,True,False
1,False,False,False,False,False,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
2,True,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
3,False,False,False,False,False,True,False,False,False,True,...,True,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False


# Drop original categorical columns
After creating dummy variables, drop the original categorical columns from the main DataFrame to avoid duplication.

In [9]:
# df4 is already a copy of the encoded features (kept for compatibility)
df4 = df3.copy()

# Feature matrix (X)
Define the feature matrix `X` as the one-hot encoded features derived from the mushroom dataset (all columns except the target `class`).

In [10]:
# Feature matrix: all one-hot encoded columns
X = df4.copy()

# Target vector (y)
Define the target variable `y` as the mushroom `class` column (mapped to 0 = edible, 1 = poisonous).

In [11]:
# Target vector: map 'class' (e=eatable, p=poisonous) to numeric labels 0/1
y = df['class'].map({'e': 0, 'p': 1})

# Train / Test split
Split the data into training and testing subsets. `test_size=0.2` keeps 20% for testing and `random_state` ensures reproducibility.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

## No Parameter
Create a Gaussian Naive Bayes classifier using default parameters. This will be trained on the data as a baseline.

In [13]:
gnb = GaussianNB()

# Execute training
Fit the GaussianNB classifier on the training data to learn parameters.

In [14]:
gnb.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


# Predictions
Use the trained model to make predictions on the test set.

In [15]:
y_pred = gnb.predict(X_test)

# Evaluation
Print classification metrics (precision, recall, f1-score) for the predictions on the test set.

In [16]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.92      0.96       842
           1       0.92      1.00      0.96       783

    accuracy                           0.96      1625
   macro avg       0.96      0.96      0.96      1625
weighted avg       0.96      0.96      0.96      1625



# Training accuracy
Compute and display the accuracy on the training set to see if the model is overfitting.

In [17]:
gnb.score(X_train, y_train)

0.9550700107708878

# Test accuracy
Compute and display the accuracy on the test set to evaluate generalization.

In [18]:
gnb.score(X_test, y_test)

0.96

# Var smoothing Parameters
Define a grid of `var_smoothing` values to search over with cross-validation. This hyperparameter controls the variance smoothing applied to avoid zero probabilities.

In [19]:
param_grid = {
    'var_smoothing': [0.00000001, 0.000000001, 0.00000001],
}

# GridSearchCV setup
Set up a grid search with 5-fold cross-validation to find the best `var_smoothing` parameter using accuracy as the scoring metric.

In [20]:
grid_search = GridSearchCV(gnb, param_grid, cv=5, scoring='accuracy', return_train_score = True, n_jobs=-1) #, verbose=10)

# Run grid search
Fit the grid search on the training data to find the best hyperparameter value. This will perform cross-validation internally.

In [21]:
grid_search.fit(X_train, y_train)

,estimator,GaussianNB()
,param_grid,"{'var_smoothing': [1e-08, 1e-09, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,priors,None


# Best parameters
Display the best hyperparameters found by the grid search.

In [22]:
grid_search.best_params_

{'var_smoothing': 1e-08}

# Best cross-validated score
Show the best cross-validated accuracy score achieved during grid search.

In [23]:
grid_search.best_score_

0.9696876887546633